In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.loader import DataLoader
from torch.utils.tensorboard import SummaryWriter
from halide_gnn_cost_model.data import PipelineDataset
from pathlib import Path
from tqdm.notebook import tqdm
import time

In [2]:
summary_writer = SummaryWriter(log_dir="../resources/runs/" + time.strftime("%Y%m%d-%H%M%S"))

In [3]:
USE_GPU = False

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: cpu


In [4]:
# Load dataset
dataset = PipelineDataset(Path("../resources/pipelines"))
data = dataset[0]  # Get the first pipeline graph
print(data.metadata)

1lines [00:00, 2468.69lines/s]
1lines [00:00, 4544.21lines/s]

<bound method HeteroData.metadata of HeteroData(
  y=[5],
  function={ x=[3, 1] },
  ast_node={ x=[16, 1] },
  loop_level={ x=[12, 1] },
  (function, called_by, function)={ edge_index=[2, 2] },
  (function, call, function)={ edge_index=[2, 2] },
  (ast_node, child_of, ast_node)={ edge_index=[2, 13] },
  (ast_node, parent_of, ast_node)={ edge_index=[2, 13] },
  (ast_node, is_expr_of, function)={ edge_index=[2, 3] },
  (function, contains_expr, ast_node)={ edge_index=[2, 3] },
  (loop_level, child_of, loop_level)={ edge_index=[2, 11] },
  (loop_level, parent_of, loop_level)={ edge_index=[2, 11] },
  (function, schedule_at, loop_level)={ edge_index=[2, 4] },
  (loop_level, schedule, function)={ edge_index=[2, 4] }
)>


In [5]:
# Train/test split
num_train = int(0.8 * len(dataset))
train_dataset, test_dataset = random_split(dataset, [num_train, len(dataset) - num_train])
len(train_dataset), len(test_dataset)

(3237, 810)

In [6]:
class PipeGCN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers=2):
        super(PipeGCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        conv = SAGEConv(-1, hidden_channels, aggr="add")
        conv.reset_parameters()
        self.convs.append(conv)
        for _ in range(num_layers - 2):
            conv = SAGEConv(-1, hidden_channels, aggr="add")
            conv.reset_parameters()
            self.convs.append(conv)
        conv = SAGEConv(-1, out_channels, aggr="add")
        conv.reset_parameters()
        self.convs.append(conv)

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [7]:
DIM_EMBEDDING = 64
gcn = PipeGCN(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'drop

In [8]:
class PipelineModel(torch.nn.Module):
    def __init__(self, gnn, out_channels, num_runtime):
        super(PipelineModel, self).__init__()
        self.ast_embedding = nn.Embedding(num_embeddings=len(dataset.ast_vocab), embedding_dim=32)
        torch.nn.init.xavier_uniform_(self.ast_embedding.weight)
        self.sched_embedding = nn.Embedding(num_embeddings=len(dataset.sched_vocab), embedding_dim=32)
        torch.nn.init.xavier_uniform_(self.sched_embedding.weight)
        self.function_gnn = gnn
        self.pipeline_lin = torch.nn.Linear(out_channels, num_runtime)
        torch.nn.init.xavier_uniform_(self.pipeline_lin.weight)

    def forward(self, data, ptr=None):
        data["ast_node"].x = self.ast_embedding(data["ast_node"].x.squeeze())
        data["loop_level"].x = self.sched_embedding(data["loop_level"].x.squeeze())
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        out = self.function_gnn(x_dict, edge_index_dict)
        # Get the feature of the pipeline node
        start = 0 if ptr is None else ptr[:-1]
        end = -1 if ptr is None else ptr[1:] - 1
        cum_sum = out["loop_level"].cumsum(dim=0)
        pipeline_feat = cum_sum[end] - cum_sum[start]
        # Predict the runtime
        log_runtime = self.pipeline_lin(pipeline_feat)
        return log_runtime

In [9]:
model = PipelineModel(gcn, DIM_EMBEDDING, 5)
model = model.to(device)
model

PipelineModel(
  (ast_embedding): Embedding(14, 32)
  (sched_embedding): Embedding(8, 32)
  (function_gnn): GraphModule(
    (convs): ModuleList(
      (0-3): 4 x ModuleDict(
        (function__called_by__function): SAGEConv(-1, 64, aggr=add)
        (function__call__function): SAGEConv(-1, 64, aggr=add)
        (ast_node__child_of__ast_node): SAGEConv(-1, 64, aggr=add)
        (ast_node__parent_of__ast_node): SAGEConv(-1, 64, aggr=add)
        (ast_node__is_expr_of__function): SAGEConv(-1, 64, aggr=add)
        (function__contains_expr__ast_node): SAGEConv(-1, 64, aggr=add)
        (loop_level__child_of__loop_level): SAGEConv(-1, 64, aggr=add)
        (loop_level__parent_of__loop_level): SAGEConv(-1, 64, aggr=add)
        (function__schedule_at__loop_level): SAGEConv(-1, 64, aggr=add)
        (loop_level__schedule__function): SAGEConv(-1, 64, aggr=add)
      )
    )
  )
  (pipeline_lin): Linear(in_features=64, out_features=5, bias=True)
)

In [ ]:
data_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
data_loader

In [11]:
for batch in data_loader:
    batch = batch.to(device)
    print(batch)
    print(model(batch, batch["loop_level"].ptr))
    break

HeteroDataBatch(
  y=[160],
  function={
    x=[180, 1],
    batch=[180],
    ptr=[33],
  },
  ast_node={
    x=[1172, 1],
    batch=[1172],
    ptr=[33],
  },
  loop_level={
    x=[682, 1],
    batch=[682],
    ptr=[33],
  },
  (function, called_by, function)={ edge_index=[2, 169] },
  (function, call, function)={ edge_index=[2, 169] },
  (ast_node, child_of, ast_node)={ edge_index=[2, 992] },
  (ast_node, parent_of, ast_node)={ edge_index=[2, 992] },
  (ast_node, is_expr_of, function)={ edge_index=[2, 180] },
  (function, contains_expr, ast_node)={ edge_index=[2, 180] },
  (loop_level, child_of, loop_level)={ edge_index=[2, 650] },
  (loop_level, parent_of, loop_level)={ edge_index=[2, 650] },
  (function, schedule_at, loop_level)={ edge_index=[2, 203] },
  (loop_level, schedule, function)={ edge_index=[2, 203] }
)
tensor([[-6.8607e+01, -3.0059e+00,  1.0582e+01, -1.4696e+00, -5.8660e+01],
        [-1.9769e+01,  1.9043e+00, -2.5863e+00,  1.5742e+01, -5.2619e+01],
        [-9.1054e+00,

In [ ]:
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.L1Loss()

for epoch in tqdm(range(200)):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(data_loader):
        optimizer.zero_grad()
        batch = batch.to(device)
        pred = model(batch, batch["loop_level"].ptr)
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), torch.log(batch.y))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        summary_writer.add_scalar("Loss/train", loss.item(), epoch * len(data_loader) + batch_idx)

  0%|          | 0/200 [00:00<?, ?it/s]

# Eval

In [ ]:
def average_runtime_error(model, data_loader):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            pred = model(batch, batch["loop_level"].ptr)
            error_rates = torch.abs(pred.reshape(-1) - batch.y) / batch.y
            total_error += torch.mean(error_rates).item()
    return total_error / len(data_loader)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, test_loader) * 100:.2f}%")

In [ ]:
# Plot predicted vs actual runtimes
import matplotlib.pyplot as plt
all_preds = []
all_trues = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = torch.exp(model(batch, batch["loop_level"].ptr))
        all_preds.append(pred.reshape(-1).cpu())
        all_trues.append(batch.y.cpu())
all_preds = torch.cat(all_preds)
all_trues = torch.cat(all_trues)
plt.scatter(all_trues, all_preds, alpha=0.5)
plt.plot([all_trues.min(), all_trues.max()], [all_trues.min(), all_trues.max()], 'r--')
plt.xscale("log")
plt.yscale("log")
plt.xlabel("True Runtime")
plt.ylabel("Predicted Runtime")
plt.title("Predicted vs True Runtimes")
plt.show()

In [ ]:
idx = 0  # Index of the sample to inspect
model.eval()
with torch.no_grad():
    sample_data = test_dataset[idx]
    sample_data = sample_data.to(device)
    pred_runtime = torch.exp(model(sample_data))
    print(f"Predicted runtime: {pred_runtime}, True runtime: {sample_data.y}")